In [ ]:
!pip install spacy

In [3]:
import spacy
import scispacy
import timeit
import pandas as pd

In [17]:
# Load the term parser
nlp = spacy.load("en_core_sci_lg")  # or "en_core_sci_md", "en_core_sci_lg", etc.

/home/jupyter-vikasni99/.local/lib/python3.12/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [15]:

def print_terms():

    text = """
    Myeloid derived suppressor cells (MDSC) are immature myeloid cells with immunosuppressive activity. 
    They accumulate in tumor-bearing mice and humans with different types of cancer, including hepatocellular carcinoma (HCC).
    """
    
    doc = nlp(text)
    
    # Extract entities
    print(len(doc.ents))
    for ent in doc.ents:
        print(ent.text, ent.label_)

11
Myeloid ENTITY
suppressor cells ENTITY
MDSC ENTITY
immature myeloid cells ENTITY
immunosuppressive activity ENTITY
accumulate ENTITY
tumor-bearing mice ENTITY
humans ENTITY
cancer ENTITY
hepatocellular carcinoma ENTITY
HCC ENTITY
0.014418550999835134 secs

9
Myeloid derived suppressor GENE_OR_GENE_PRODUCT
cells CELL
MDSC CELL
myeloid cells CELL
mice ORGANISM
humans ORGANISM
cancer CANCER
hepatocellular carcinoma CANCER
HCC CANCER
0.019169123959727585 secs


In [18]:


exec_time = timeit.timeit(print_terms,number=1)
print(exec_time, "secs")

11
Myeloid ENTITY
suppressor cells ENTITY
MDSC ENTITY
immature myeloid cells ENTITY
immunosuppressive activity ENTITY
accumulate ENTITY
tumor-bearing mice ENTITY
humans ENTITY
cancer ENTITY
hepatocellular carcinoma ENTITY
HCC ENTITY
0.01960328791756183 secs


In [ ]:
import pandas as pd

md_df = pd.read_parquet('OSD-100_Mmus_C57-6J_EYE_FLT_Rep1_M23_archs4_top10hits_metadata.parquet')
print(md_df.head())

In [7]:
md_df.columns

Index(['gsm', 'score', 'gse', 'title', 'source_name', 'characteristics',
       'tissue', 'species', 'geo_summary', 'geo_design', 'pubmed_ids',
       'archs4_index', 'geo_platform_biopython', 'geo_taxon_biopython',
       'geo_entry_type_biopython', 'geo_gds_type_biopython',
       'geo_pdat_biopython', 'geo_n_samples_biopython',
       'geo_ftp_link_biopython', 'pubmed_title_biopython',
       'pubmed_journal_biopython', 'pubmed_pub_date_biopython',
       'pubmed_doi_biopython'],
      dtype='str')

In [8]:
md_df.iloc[1]['pubmed_title_biopython']

'Tissue-specific modifier alleles determine Mertk loss-of-function traits.'

In [9]:
md_df.head()

,gsm,score,gse,title,source_name,characteristics,tissue,species,geo_summary,geo_design,...,geo_taxon_biopython,geo_entry_type_biopython,geo_gds_type_biopython,geo_pdat_biopython,geo_n_samples_biopython,geo_ftp_link_biopython,pubmed_title_biopython,pubmed_journal_biopython,pubmed_pub_date_biopython,pubmed_doi_biopython
0,GSM6431263,0.997020,GSE210492,"posterior eyecup, Wt, 17 months, 7-2",posterior eyecup,"cell line: posterior eyecup,cell type: female,...",Other,Mus musculus,To characterize female aged Efemp1 R345W knock...,Gene expression profiling analysis of RNA-seq ...,...,Mus musculus,GSE,Expression profiling by high throughput sequen...,2022/08/10,93,ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE210nn...,,,,
1,GSM6204794,0.995838,GSE205070,"RPE, MertkOV1, P25, Biol rep 6",RPE,"tissue: RPE,strain: C57BL/6J,cell type: epithe...",Eye,Mus musculus,Knockout (KO) mouse models play critical roles...,We performed gene expression profiling analysi...,...,Mus musculus,GSE,Expression profiling by high throughput sequen...,2022/08/26,30,ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE205nn...,Tissue-specific modifier alleles determine Mer...,eLife,2022 Aug 15,10.7554/eLife.80530
2,GSM6431262,0.995728,GSE210492,"posterior eyecup, Wt, 17 months, 7-1",posterior eyecup,"cell line: posterior eyecup,cell type: female,...",Other,Mus musculus,To characterize female aged Efemp1 R345W knock...,Gene expression profiling analysis of RNA-seq ...,...,Mus musculus,GSE,Expression profiling by high throughput sequen...,2022/08/10,93,ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE210nn...,Complement factor B is critical for sub-RPE de...,Human molecular genetics,2023 Jan 6,10.1093/hmg/ddac187
3,GSM6431234,0.995604,GSE210492,"posterior eyecup, Wt, 9 months, 3-4",posterior eyecup,"cell line: posterior eyecup,cell type: female,...",Other,Mus musculus,To characterize female aged Efemp1 R345W knock...,Gene expression profiling analysis of RNA-seq ...,...,Mus musculus,GSE,Expression profiling by high throughput sequen...,2022/08/10,93,ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE210nn...,,,,
4,GSM4256053,0.995411,GSE143281,P30 CTL1,Retina,"strain: C57BL/6,tissue: Retina,genotype: control",Eye,Mus musculus,"Ubiquitously expressed transcript (UXT), a sma...",Total RNA in mouse retina after conditional kn...,...,Mus musculus,GSE,Expression profiling by high throughput sequen...,2020/08/07,16,ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE143nn...,Mice deficient in UXT exhibit retinitis pigmen...,Autophagy,2021 Aug,10.1080/15548627.2020.1796015


In [ ]:
###########

In [30]:
#for each entry reduce them down to the terms with SpaCy and then flatten it 
#add column to df to incl. the term words

md_df['spacey_terms'] = md_df.apply(lambda _: [], axis=1)

def _add_spacey_terms():
    
    for row in md_df.itertuples():
         #print(row.geo_summary)
         doc = nlp(row.geo_summary) #returns a doc object
         #extract terms 
         for ent in doc.ents:
             row.spacey_terms.append(ent.text)


In [31]:
exec_time = timeit.timeit(_add_spacey_terms,number=1)
print(exec_time, "secs")

0.28130936704110354 secs


In [32]:
md_df.head()

,gsm,score,gse,title,source_name,characteristics,tissue,species,geo_summary,geo_design,...,geo_entry_type_biopython,geo_gds_type_biopython,geo_pdat_biopython,geo_n_samples_biopython,geo_ftp_link_biopython,pubmed_title_biopython,pubmed_journal_biopython,pubmed_pub_date_biopython,pubmed_doi_biopython,spacey_terms
0,GSM6431263,0.997020,GSE210492,"posterior eyecup, Wt, 17 months, 7-2",posterior eyecup,"cell line: posterior eyecup,cell type: female,...",Other,Mus musculus,To characterize female aged Efemp1 R345W knock...,Gene expression profiling analysis of RNA-seq ...,...,GSE,Expression profiling by high throughput sequen...,2022/08/10,93,ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE210nn...,,,,,"[female, aged, Efemp1, R345W, Efemp1ki/ki, mic..."
1,GSM6204794,0.995838,GSE205070,"RPE, MertkOV1, P25, Biol rep 6",RPE,"tissue: RPE,strain: C57BL/6J,cell type: epithe...",Eye,Mus musculus,Knockout (KO) mouse models play critical roles...,We performed gene expression profiling analysi...,...,GSE,Expression profiling by high throughput sequen...,2022/08/26,30,ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE205nn...,Tissue-specific modifier alleles determine Mer...,eLife,2022 Aug 15,10.7554/eLife.80530,"[Knockout (KO), mouse models, biological proce..."
2,GSM6431262,0.995728,GSE210492,"posterior eyecup, Wt, 17 months, 7-1",posterior eyecup,"cell line: posterior eyecup,cell type: female,...",Other,Mus musculus,To characterize female aged Efemp1 R345W knock...,Gene expression profiling analysis of RNA-seq ...,...,GSE,Expression profiling by high throughput sequen...,2022/08/10,93,ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE210nn...,Complement factor B is critical for sub-RPE de...,Human molecular genetics,2023 Jan 6,10.1093/hmg/ddac187,"[female, aged, Efemp1, R345W, Efemp1ki/ki, mic..."
3,GSM6431234,0.995604,GSE210492,"posterior eyecup, Wt, 9 months, 3-4",posterior eyecup,"cell line: posterior eyecup,cell type: female,...",Other,Mus musculus,To characterize female aged Efemp1 R345W knock...,Gene expression profiling analysis of RNA-seq ...,...,GSE,Expression profiling by high throughput sequen...,2022/08/10,93,ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE210nn...,,,,,"[female, aged, Efemp1, R345W, Efemp1ki/ki, mic..."
4,GSM4256053,0.995411,GSE143281,P30 CTL1,Retina,"strain: C57BL/6,tissue: Retina,genotype: control",Eye,Mus musculus,"Ubiquitously expressed transcript (UXT), a sma...",Total RNA in mouse retina after conditional kn...,...,GSE,Expression profiling by high throughput sequen...,2020/08/07,16,ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE143nn...,Mice deficient in UXT exhibit retinitis pigmen...,Autophagy,2021 Aug,10.1080/15548627.2020.1796015,"[expressed, UXT, chaperone-like protein, expre..."


In [33]:
md_df.iloc[0]['spacey_terms']

['female',
 'aged',
 'Efemp1',
 'R345W',
 'Efemp1ki/ki',
 'mice',
 'model',
 'Doyne',
 'retinal',
 'dystrophy/malattia leventinese',
 'RNA sequencing',
 'expression',
 'gene set enrichment analysis analysis',
 'data',
 'RNA-seq',
 'neural retina',
 'posterior eyecups',
 'whole eye',
 'lens',
 'retina',
 'Efemp1+/+',
 'mice']

In [39]:
#flatten spacey terms across all hits
flat_terms = [item for lst in md_df["spacey_terms"] for item in lst]
flat_terms

['female',
 'aged',
 'Efemp1',
 'R345W',
 'Efemp1ki/ki',
 'mice',
 'model',
 'Doyne',
 'retinal',
 'dystrophy/malattia leventinese',
 'RNA sequencing',
 'expression',
 'gene set enrichment analysis analysis',
 'data',
 'RNA-seq',
 'neural retina',
 'posterior eyecups',
 'whole eye',
 'lens',
 'retina',
 'Efemp1+/+',
 'mice',
 'Knockout (KO)',
 'mouse models',
 'biological processes',
 'disease-associated',
 'disease-resistant',
 'traits',
 'gene KO',
 'mice',
 'phenotypes',
 'molecular role',
 'gene',
 'biological process',
 'biological process',
 'causally',
 'trait',
 'pathological',
 'traits',
 'associated with',
 'Mertk KO',
 'MERTK',
 'receptor tyrosine kinase',
 'phagocytosis',
 'apoptotic cells',
 'cellular debris',
 'early-onset',
 'severe',
 'retinal degeneration',
 'failed',
 'phagocytosis',
 'photoreceptor',
 'outer segments',
 'retinal',
 'pigment epithelia',
 'anti-tumor immunity',
 'failure',
 'macrophages',
 'dispose cancer',
 'cell corpses',
 'pro-inflammatory',
 'micro